In [1]:
import numpy as np
import matplotlib as mt
import matplotlib.pyplot as plt

In [2]:
selected = np.loadtxt('../3_chains/selected.txt', dtype=str)

In [3]:
selected.shape

(539,)

In [24]:
def read_pdb(pdb, path='../3_chains/chains', 
            entries=['ATOM'], atoms=['N','CA','C'],
            mini=25, maxi=210):
    out = []
    for l in open(f'{path}/{pdb}.pdb', 'r'):
        if any([i in l[:6] for i in entries]):
            if mini <= int(l[23:26].strip()) <= maxi:
                if l[13:16].strip() in atoms:
                    out.append(l)
    return out

def remove_repeats(out):
    out2 = []
    present = {}
    for l in out:
        resi = int(l[23:26].strip())
        atom = l[13:16].strip()
        #
        if resi not in present.keys():
            present[resi]=[]
        #
        if atom not in present[resi]:
            present[resi].append(atom)
            out2.append(l)
        #
    return out2


def check_atoms(out, atoms=['C','CA','N']):
    anames = {}
    for l in out:
        resi = int(l[23:26].strip())
        if resi not in anames.keys():
            anames[resi]=[]
        anames[resi].append(l[13:16].strip())
    #
    for resi,a in anames.items():
        if sorted(a) == atoms:
            anames[resi] = True
        else:
            anames[resi] = False
    #
    return all(anames.values())

def write_pdb(out, ofile, path):
    ofile = open(f'{path}/{ofile}.pdb', 'w')
    for l in out:
        ofile.write(l)
    ofile.close()

#### check - if all bb atoms are present

In [16]:
trues=[]
for pdb in selected:
    out=read_pdb(pdb)
    trues.append(check_atoms(out))
print(np.sum(trues), np.sum(trues)/len(trues))

369 0.6846011131725418


In [18]:
trues=[]
for pdb in selected:
    out=read_pdb(pdb)
    out = remove_repeats(out)
    trues.append(check_atoms(out))
print(np.sum(trues), np.sum(trues)/len(trues))

539 1.0


#### write chains

In [26]:
for pdb in selected:
    out = read_pdb(pdb)
    out = remove_repeats(out)
    write_pdb(out, pdb, '0_selected_chains')